In [76]:
import numpy as np
import weakref
class Variable:
    def __init__(self, data, name=None):
        if data is not None:
            if not isinstance(data, np.ndarray):
                raise TypeError('{} is not supported'.format(type(data)))
        self.data = data
        self.name = name
        self.grad = None
        self.creator = None
        self.generation = 0
    def set_creator(self, func):
        self.creator = func
        self.generation = func.generation + 1
    def backward(self,retain_grad=False):
        if self.grad is None:
            self.grad = np.ones_like(self.data)
        funcs = []
        seen_set = set()
        def add_func(f):
            if f not in seen_set:
                funcs.append(f)
                seen_set.add(f)
                funcs.sort(key=lambda x: x.generation)
        add_func(self.creator)
        while funcs:
            f = funcs.pop()
            # x, y = f.input, f.output
            # gys = [output.grad for output in f.outputs]
            gys = [output().grad for output in f.outputs]
            # x.grad = f.backward(y.grad)
            gxs = f.backward(*gys)
            if not isinstance(gxs, tuple):
                gxs = (gxs,)
            for x, gx in zip(f.inputs, gxs):
                if x.grad is None:
                    x.grad = gx
                else:
                    x.grad = x.grad + gx
                # if x.creator is not None:
                if x.creator is not None:
                    add_func(x.creator)
            if not retain_grad:
                for y in f.outputs:
                    y().grad = None
    def cleargrad(self):
        self.grad = None
    @property
    def shape(self):
        return self.data.shape
    @property
    def size(self):
        return self.data.size
    @property
    def ndim(self):
        return self.data.ndim
    @property
    def dtype(self):
        return self.data.dtype
    def __len__(self):
        return len(self.data)
    def __repr__(self):
        if self.data is None:
            return 'variable(None)'
        p = str(self.data).replace('\n', '\n' + ' '*9)
        return 'variable(' + p + ')'
    def __mul__(self, other):
        return mul(self, other)
    def __add__(self, other):
        return add(self, other)
def as_array(x):
    if np.isscalar(x):
        x = np.array(x)
    return x        

def as_variable(obj):
    if isinstance(obj, Variable):
        return obj
    return Variable(obj)
class Function:
    def __call__(self, *inputs):
        inputs = [as_variable(input) for input in inputs]
        # x = input.data
        self.generation = 0
        xs = [x.data for x in inputs]
        ys = self.forward(*xs)
        if not isinstance(ys, tuple):
            ys = (ys,)
        #output = Variable(as_array(y))
        outputs = [Variable(as_array(y)) for y in ys]
        if Config.enable_backprop:
            
            self.generation = max([input.generation for input in inputs])
        
            # output.set_creator(self)
            for output in outputs:
                output.set_creator(self)
        
        #remember
        self.inputs = inputs
        self.outputs = [weakref.ref(output) for output in outputs]
        return outputs if len(outputs) > 1 else outputs[0]
    def forward(self, x):
        raise NotImplementedError()
        # return 当前模块的原函数
    def backward(self, gy):
        raise NotImplementedError()
        # x = self.input.data
        # gx = 当前模块的导数(x) * gy
        # return gx
class Config:
    enable_backprop = True
class Square(Function):
    def forward(self, x):
        return x**2
    def backward(self, gy):
        x = self.inputs[0].data
        gx = 2 * x * gy
        return gx
class Exp(Function):
    def forward(self, x):
        return np.exp(x)
    def backward(self, gy):
        x = self.input.data
        gx = np.exp(x) * gy
        return gx
def square(x):
    return Square()(x)
def exp(x):
    return Exp()(x)

class Add(Function):
    def forward(self, x0, x1):
        #print(type(xs))
        x1 = as_array(x1)
        y = x0 + x1
        return y
    def backward(self, gy):
        return gy, gy
def add(x0, x1):
    return Add()(x0,x1)
class Mul(Function):
    def forward(self,x0, x1):
        x1 = as_array(x1)
        y = x0 * x1
        return y
    def backward(self, gy):
        x0, x1 = self.inputs[0].data, self.inputs[1].data
        return x1 * gy, x0 * gy
def mul(x0, x1):
    return Mul()(x0, x1)


In [78]:
Variable.__add__ = add
Variable.__radd__ = add
Variable.__mul__ = mul
Variable.__rmul__ = mul
x = Variable(np.array(2.))
y =  x + 3
print(y)

TypeError: <class 'int'> is not supported

In [65]:
a = Variable(np.array(3.0))
b = Variable(np.array(2.0))
c = Variable(np.array(1.0))

y = a*b + c
y.backward()
print(y)
print(a.grad)
print(b.grad)

variable(7.0)
2.0
3.0


In [57]:
x = np.arange(6,dtype=np.float32).reshape(2,-1) + 1
x = Variable(x)
print(x.shape)
print(x.ndim)
print(x.size)
print(x.dtype)
print(len(x))
print(x)


(2, 3)
2
6
float32
2
variable([[1. 2. 3.]
          [4. 5. 6.]])
[[1. 2. 3.]
          [4. 5. 6.]]


In [13]:
import contextlib

@contextlib.contextmanager
def config_test():
    print('start')
    try:
        yield
    finally:
        print('done')
        
with config_test():
    print('process')

start
process
done


In [24]:
import contextlib
@contextlib.contextmanager
def using_config(name, value):
    print('start')
    old_value = getattr(Config, name)
    setattr(Config, name, value)
    try:
        yield
    finally:
        setattr(Config, name, old_value)
        print('done')
        
def no_grad():
    return using_config("enable_backprop", False)

In [25]:
with no_grad():
    x = Variable(np.array(2.0))
    y = square(x)

start
done


In [171]:
with open('sample.txt', 'w') as f:
    f.write('hello world!')

In [ ]:
Config.enable_backprop = True
x = Variable(np.ones((100,100,100)))
y = square(square(square(x)))
y.backward()

Config.enable_backprop = False
x = Variable(np.ones((100,100,100)))
y = square(square(square(x)))

In [169]:
x0 = Variable(np.array(1.0))
x1 = Variable(np.array(1.0))
t= add(x0,x1)
y = add(x0, t)
y.backward()
print(y.grad, t.grad)
print(x0.grad, x1.grad)

1.0 1.0
2.0 1.0


In [168]:
for i in range(10):
    x = Variable(np.random.randn(10000))
    y = square(square(square(x)))

In [165]:
import weakref
import numpy as np
a = np.array([1,2,3])
b = weakref.ref(a)
b()
a = None
b

<weakref at 0x000001CED0965A90; dead>

In [ ]:
class obj:
    pass
def f(x):
    print(x)
a = obj()
f(a)
a =None

In [159]:
x = Variable(np.array(2.0))
a = square(x)
b = square(a)
c = square(a)
y = add(b,c)

y.backward()
print(y.data)
print(x.grad)

32.0
64.0


In [150]:
generations = [2, 0, 1, 4, 2]
funcs = []
for g in generations:
    f = Function()
    f.generation = g
    funcs.append(f)
[f.generation for f in funcs]

[2, 0, 1, 4, 2]

In [152]:
funcs.sort(key=lambda x: x.generation)
[f.generation for f in funcs]
f = funcs.pop()
f.generation

4

In [147]:
x = Variable(np.array(3.))
y = add(x,x)
y.backward()
print('x.grad', x.grad)

x.cleargrad()
y = add(x, add(x,x))
y.backward()
print(x.grad)

x.grad 2.0
3.0


In [121]:
x = Variable(np.array(2.0))
print(x.data)
y = Variable(np.array(3.0))
print(y.data)
print(type(y))
z = add(square(x), square(y))

z.backward()
print(z.data)
print(x.grad)
print(y.grad)

2.0
3.0
<class '__main__.Variable'>
<class '__main__.Add'>
<class '__main__.Add'>
[array(1.)]
<class '__main__.Square'>
[array(1.)]
<class '__main__.Square'>
[array(1.)]
13.0
4.0
6.0


In [27]:
import unittest
class SquareTest(unittest.TestCase):
    def test_forward(self):
        x = Variable(np.array(2.0))
        y = square(x)
        expected = np.array(4.0)
        self.assertEqual(y.data, expected)

In [47]:
def numerical_diff(f, x, eps=1e-4):
    x1, x0 = Variable(x.data + eps), Variable(x.data - eps)
    y1, y0 = f(x1), f(x0)
    return (y1.data-y0.data) / (2 * eps )

In [49]:
x = Variable(np.array([2]))
numerical_diff(square,x, 0.1)

array([4.])

In [14]:
unittest.main()

E
ERROR: C:\Users\Lenovo\AppData\Roaming\jupyter\runtime\kernel-10f9e3f9-b8f1-43d8-9cf6-db658e5f7354 (unittest.loader._FailedTest)
----------------------------------------------------------------------
AttributeError: module '__main__' has no attribute 'C:\Users\Lenovo\AppData\Roaming\jupyter\runtime\kernel-10f9e3f9-b8f1-43d8-9cf6-db658e5f7354'

----------------------------------------------------------------------
Ran 1 test in 0.001s

FAILED (errors=1)


SystemExit: True

D:\miniconda\envs\d2l\lib\site-packages\IPython\core\interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [6]:
#test
A = Square()
B = Exp()
C = Square()

x = Variable(np.array(0.5))
a = A(x)
b = B(a)
y = C(b)

y.backward()

print(x.grad)

0.5 <class 'numpy.ndarray'>
0.25 <class 'numpy.float64'>


TypeError: <class 'numpy.float64'> is not supported

In [3]:
x = np.array([1.0])
y = x**2
print(type(x), x.ndim)
print(type(y))

<class 'numpy.ndarray'> 1
<class 'numpy.ndarray'>


In [4]:
x = np.array(1.0)
y = x**2
print(type(x), x.ndim)
print(type(y))

<class 'numpy.ndarray'> 0
<class 'numpy.float64'>


In [96]:
import numpy as np


class Variable:
    def __init__(self, data):
        if data is not None:
            if not isinstance(data, np.ndarray):
                raise TypeError('{} is not supported'.format(type(data)))

        self.data = data
        self.grad = None
        self.creator = None

    def set_creator(self, func):
        self.creator = func

    def backward(self):
        if self.grad is None:
            self.grad = np.ones_like(self.data)

        funcs = [self.creator]
        while funcs:
            f = funcs.pop()
            gys = [output.grad for output in f.outputs]
            gxs = f.backward(*gys)
            if not isinstance(gxs, tuple):
                gxs = (gxs,)

            for x, gx in zip(f.inputs, gxs):
                x.grad = gx

                if x.creator is not None:
                    funcs.append(x.creator)


def as_array(x):
    if np.isscalar(x):
        return np.array(x)
    return x


class Function:
    def __call__(self, *inputs):
        xs = [x.data for x in inputs]
        ys = self.forward(*xs)
        if not isinstance(ys, tuple):
            ys = (ys,)
        outputs = [Variable(as_array(y)) for y in ys]

        for output in outputs:
            output.set_creator(self)
        self.inputs = inputs
        self.outputs = outputs
        return outputs if len(outputs) > 1 else outputs[0]

    def forward(self, xs):
        raise NotImplementedError()

    def backward(self, gys):
        raise NotImplementedError()


class Square(Function):
    def forward(self, x):
        y = x ** 2
        return y

    def backward(self, gy):
        x = self.inputs[0].data
        gx = 2 * x * gy
        return gx


def square(x):
    f = Square()
    return f(x)


class Add(Function):
    def forward(self, x0, x1):
        y = x0 + x1
        return y

    def backward(self, gy):
        return gy, gy


def add(x0, x1):
    return Add()(x0, x1)


x = Variable(np.array(2.0))
y = Variable(np.array(3.0))

z = add(square(x), square(y))
z.backward()
print(z.data)
print(x.grad)
print(y.grad)

13.0
4.0
6.0
